# Model Training Notebook

This notebook trains a Random Forest model on the gut survey data, builds a preprocessing pipeline, and saves the trained model.

In [41]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np
import re

## 1. Load and Inspect Data

In [43]:
# 1.1 Load your collected data
df = pd.read_csv("synthetic_bowel_movements.csv")
df.head()

,What is your age?,How would you rate the typical smell intensity of your stool?,"On average, how many hours of sleep do you get per night?",Timestamp,What is your gender?,Height (cm),Weight (kg),Hydration Level,How active are you physically on average?,Are you currently taking any medication that affects digestion or bowel movements?,...,How many meals with greasy or fried food do you eat per week?,"Do you regularly consume dairy products (milk, cheese, yogurt)?","On average, how many servings of processed food do you eat per day?","On average, how many servings of fruits and vegetables do you eat per day?",What type of toilet paper or wiping method do you usually use?,How would you describe your typical stool consistency?,What is the most common color of your stool?,"How many times do you go to the toilet for the number ""2"" in a week?","How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?","On average, how many wipes or sheets of toilet paper do you use per bowel movement?"
0,29,4,7,02/05/2025 22:48:43,Male,180,95,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,5,3-5,8
1,13,4,8,02/05/2025 23:02:38,Male,178,69,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,0-2,Yes,0-2,0-2,3-ply paper,Soft,Brown,5-7,5 and more,24
2,20,2,6,02/05/2025 22:51:02,Male,161,74,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5 and more,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,2-3,0-2,7
3,29,2,8,02/05/2025 23:02:48,Male,163,70,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,0-2,Yes,3-4,0-2,3-ply paper,Firm and smooth,Brown,14,0-2,Like 10 wipes
4,18,1,4,02/05/2025 22:50:20,Male,181,69,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,No,0-2,0-2,2-ply paper,Hard and lumpy,Brown,5,5 and more,3


In [44]:
column_mapping = {
    "What is your age?": "age",
    "How would you rate the typical smell intensity of your stool?": "smell_intensity",
    "On average, how many hours of sleep do you get per night?": "sleep_hours",
    "Timestamp": "timestamp",
    "What is your gender?": "gender",
    "Height (cm)": "height_cm",
    "Weight (kg)": "weight_kg",
    "Hydration Level": "hydration_level",
    "How active are you physically on average?": "activity_level",
    "Are you currently taking any medication that affects digestion or bowel movements?": "meds_affecting_gut",
    "How much dietary fibers do you eat daily?": "fiber_grams",
    "How much fat do you consume daily?": "fat_grams",
    "How spicy is your typical diet?": "spiciness",
    " How many meals with greasy or fried food do you eat per week?": "weekly_greasy_meals",
    "Do you regularly consume dairy products (milk, cheese, yogurt)?": "dairy_freq",
    "On average, how many servings of processed food do you eat per day?": "processed_servings",
    "On average, how many servings of fruits and vegetables do you eat per day?": "fv_servings",
    "What type of toilet paper or wiping method do you usually use?": "toilet_method",
    "How would you describe your typical stool consistency?": "stool_consistency",
    "What is the most common color of your stool?": "stool_color",
    "How many times do you go to the toilet for the number \"2\" in a week?": "weekly_bms",
    "How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?": "caffeinated_beverages_per_day",
    "On average, how many wipes or sheets of toilet paper do you use per bowel movement?": "wipes_per_bm"
}

   

   
# Apply the mapping
df.rename(columns=column_mapping, inplace=True)

In [46]:
df.head()

,age,smell_intensity,sleep_hours,timestamp,gender,height_cm,weight_kg,hydration_level,activity_level,meds_affecting_gut,...,weekly_greasy_meals,dairy_freq,processed_servings,fv_servings,toilet_method,stool_consistency,stool_color,weekly_bms,caffeinated_beverages_per_day,wipes_per_bm
0,29.0,4.0,7.0,02/05/2025 22:48:43,Male,180.0,95.0,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,1.0,Yes,1.0,1.0,2-ply paper,Firm and smooth,Brown,5.0,3-5,8
1,13.0,4.0,8.0,02/05/2025 23:02:38,Male,178.0,69.0,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,1.0,Yes,1.0,1.0,3-ply paper,Soft,Brown,6.0,5 and more,24
2,20.0,2.0,6.0,02/05/2025 22:51:02,Male,161.0,74.0,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5.0,Yes,1.0,1.0,2-ply paper,Firm and smooth,Brown,2.5,0-2,7
3,29.0,2.0,8.0,02/05/2025 23:02:48,Male,163.0,70.0,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,1.0,Yes,3.5,1.0,3-ply paper,Firm and smooth,Brown,14.0,0-2,Like 10 wipes
4,18.0,1.0,4.0,02/05/2025 22:50:20,Male,181.0,69.0,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,1.0,No,1.0,1.0,2-ply paper,Hard and lumpy,Brown,5.0,5 and more,3


## Data cleaning

In [47]:
# 4. Clean numeric columns
numeric_cols = ['age', 'height_cm', 'weight_kg', 'sleep_hours']
for col in numeric_cols:
    # strip non‐digits (e.g. '90kg = bulking' → '90')
    df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)

# 5. Map Yes/No to boolean
#df['meds_affecting_gut'] = df['meds_affecting_gut'].map({'Yes': True, 'No': False})
#df['dairy_freq'] = df['dairy_freq'].map({'Yes': 'regular', 'No': 'none', 'Occasionally': 'occasional'})

# 6. Helper to parse ranges like "3-4", "5 and more", "0-2"
def parse_range(val):
    if pd.isna(val): 
        return np.nan
    s = str(val)
    if 'and more' in s:
        return float(re.search(r'(\d+)', s).group(1))
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    # fallback: extract single number
    m2 = re.search(r'(\d+)', s)
    return float(m2.group(1)) if m2 else np.nan

# 7. Apply range parser to these columns
range_cols = [
    'smell_intensity', 'weekly_greasy_meals', 'processed_servings', 
    'fv_servings', 'weekly_bms', 'wipes_per_bm'
]
for col in range_cols:
    df[col] = df[col].apply(parse_range)

# 8. Clean categorical columns if needed (e.g. hydration, spiciness, activity)
#    You can ordinal‐encode or OneHotEncode later in your pipeline.
#df['hydration_level'] = df['hydration_level'] \
#    .str.replace(r'[^0-9\-\+]', '', regex=True)
#df['activity_level'] = df['activity_level'].str.strip()

# 9. Final sanity check
print(df.dtypes)
df.head()

age                              float64
smell_intensity                  float64
sleep_hours                      float64
timestamp                         object
gender                            object
height_cm                        float64
weight_kg                        float64
hydration_level                   object
activity_level                    object
meds_affecting_gut                object
fiber_grams                       object
fat_grams                         object
spiciness                         object
weekly_greasy_meals              float64
dairy_freq                        object
processed_servings               float64
fv_servings                      float64
toilet_method                     object
stool_consistency                 object
stool_color                       object
weekly_bms                       float64
caffeinated_beverages_per_day     object
wipes_per_bm                     float64
dtype: object


,age,smell_intensity,sleep_hours,timestamp,gender,height_cm,weight_kg,hydration_level,activity_level,meds_affecting_gut,...,weekly_greasy_meals,dairy_freq,processed_servings,fv_servings,toilet_method,stool_consistency,stool_color,weekly_bms,caffeinated_beverages_per_day,wipes_per_bm
0,29.0,4.0,7.0,02/05/2025 22:48:43,Male,180.0,95.0,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,1.0,Yes,1.0,1.0,2-ply paper,Firm and smooth,Brown,5.0,3-5,8.0
1,13.0,4.0,8.0,02/05/2025 23:02:38,Male,178.0,69.0,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,1.0,Yes,1.0,1.0,3-ply paper,Soft,Brown,6.0,5 and more,24.0
2,20.0,2.0,6.0,02/05/2025 22:51:02,Male,161.0,74.0,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5.0,Yes,1.0,1.0,2-ply paper,Firm and smooth,Brown,2.0,0-2,7.0
3,29.0,2.0,8.0,02/05/2025 23:02:48,Male,163.0,70.0,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,1.0,Yes,3.0,1.0,3-ply paper,Firm and smooth,Brown,14.0,0-2,10.0
4,18.0,1.0,4.0,02/05/2025 22:50:20,Male,181.0,69.0,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,1.0,No,1.0,1.0,2-ply paper,Hard and lumpy,Brown,5.0,5 and more,3.0


## 2. Split Features and Target

In [ ]:
# 1.2 Split into X/y
X = df.drop("wipes_per_bm", axis=1)  # e.g. 'next_bowel_movement_in_hours'
y = df["wipes_per_bm"]
X.head(), y.head()

KeyError: "['target_label'] not found in axis"

## 3. Build Preprocessing Pipeline

In [ ]:
# 1.3 Define numeric and categorical features
numeric_features = [
    "age", "height", "weight", "hydration_level_liters", 
    "fiber_grams", "fat_grams", "weekly_greasy_meals", 
    "processed_servings", "fv_servings", "weekly_bms", "sheets_per_bm"
]
categorical_features = [
    "gender", "activity_level", "meds_affecting_gut",
    "spiciness", "dairy_freq", "toilet_method",
    "stool_consistency", "stool_color", "smell_intensity"
]

# Transformations
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## 4. Create and Train the Pipeline

In [ ]:
# 1.4 Create full pipeline with an estimator
pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the pipeline
pipeline.fit(X, y)

## 5. Save the Trained Model

In [ ]:
# 1.5 Serialize the trained pipeline
joblib.dump(pipeline, "model.pkl")
print("Model trained and saved to model.pkl")